# Understanding the statin evidence figures

This notebook redraws published figures and repeats three aggregate meta-analyses from the **2022 USPSTF review**. It explains the overview, the mortality sensitivity analysis, and the harms comparison.

Follow the [setup and extraction instructions](../README.md#reproduce-the-figures) before running all cells with the project's `.venv` Python environment. Extracted data and execution outputs are generated locally and excluded from Git. Once the inputs exist, running the notebook needs no downloads. Exports go to `analysis/notebook_output/`; the analysis and plotting functions are shared with the scripts.

The README's versioned figure assets are in `figures/`. Clear cell outputs again before staging changes to this notebook.


In [ ]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import Image, display

# Find the project whether the kernel starts in the root or in analysis/.
ROOT = next(
    (path for path in [Path.cwd(), *Path.cwd().parents]
     if (path / "analysis" / "meta_analysis.py").is_file()),
    None,
)
if ROOT is None:
    raise FileNotFoundError("Start the notebook from this project or its analysis/ folder.")

ANALYSIS = ROOT / "analysis"
DATA = ANALYSIS / "data"
sys.path.insert(0, str(ANALYSIS))

from meta_analysis import analyze_counts, leave_one_out, sensitivity
import run_analysis as plots

PLOT_DIR = ANALYSIS / "notebook_output"
PLOT_DIR.mkdir(exist_ok=True)
plots.OUT = PLOT_DIR


## Where the data come from

These are **published aggregate numbers**, not patient records.

| Figure content | Source and local data | How obtained |
| --- | --- | --- |
| Overview A and B | [CTT 2012, Figures 2 and 5](https://pmc.ncbi.nlm.nih.gov/articles/PMC3437972/); risk effects (`data/ctt_2012_risk_effects.csv`, generated locally), absolute projections (`data/ctt_2012_absolute_projections.csv`, generated locally) | Transcribed printed estimates and numerical labels. |
| Overview C and D; mortality analysis | [Chou et al. 2022, Figures 3 and 4](https://doi.org/10.1001/jama.2022.12138); trial counts (`data/uspstf_2022_trial_counts.csv`, generated locally), published summaries (`data/uspstf_2022_pooled_outcomes.csv`, generated locally) | Trial counts parsed from PDF page 10 and checked against printed totals. |
| Muscle symptoms | [CTT 2022, Figure 3](https://pmc.ncbi.nlm.nih.gov/articles/PMC7613583/); rates (`data/ctt_2022_muscle_key_rates.csv`, generated locally) | Transcribed rounded rates and excess rates. |
| Diabetes excess | [CTT 2024, Figure 2](https://pmc.ncbi.nlm.nih.gov/articles/PMC7615958/); projections (`data/ctt_2024_diabetes_excess.csv`, generated locally) | Transcribed modeled excess rates. |

Curve heights were not digitized. The extraction manifest (`data/extraction_manifest.json`, generated locally) records source files and hashes. AHRQ's listed workbook could not be retrieved and was not used.


In [ ]:
counts = pd.read_csv(DATA / "uspstf_2022_trial_counts.csv")
risk_effects = pd.read_csv(DATA / "ctt_2012_risk_effects.csv")
absolute_projections = pd.read_csv(DATA / "ctt_2012_absolute_projections.csv")
published = pd.read_csv(DATA / "uspstf_2022_pooled_outcomes.csv")

# Example: each row contains one trial's results for one outcome.
mortality = counts.loc[counts["outcome"] == "All-cause mortality"].copy()
display(mortality[[
    "trial", "statin_events", "statin_n", "control_events", "control_n"
]].head(6))


## Recalculate the pooled results

For each trial, the code computes the risk ratio from events divided by the arm total, then pools log risk ratios using inverse-variance random effects with the **DerSimonian–Laird** method and normal 95% confidence intervals.

A study with no events in either arm contributes no risk-ratio estimate. A single zero-event arm receives a 0.5 correction to all four cells of that study. This leaves **17 contributing mortality estimates from 18 trial rows**. These are risk ratios over trial follow-up, not reconstructed survival models.


In [ ]:
summary = {
    outcome: analyze_counts(group).as_dict()
    for outcome, group in counts.groupby("outcome", sort=False)
}

recalculated = pd.DataFrame([
    {
        "Outcome": outcome,
        "Contributing estimates": result["trials"],
        "RR (95% CI)": (
            f'{result["rr"]:.3f} '
            f'({result["ci_low"]:.3f}–{result["ci_high"]:.3f})'
        ),
        "I² (%)": round(result["i2_percent"], 1),
    }
    for outcome, result in summary.items()
])
display(recalculated)


## The overview figure

- **A:** Published CTT effects with and without previous vascular disease. Risk-stratum intervals are **99%**; overall summaries use **95%**. Values below 1 favor statins for these undesirable outcomes.
- **B:** Published five-year projections of events prevented per 1,000, by baseline risk and LDL reduction. Lines join model outputs; they are not a new regression or observed event counts.
- **C:** Published primary prevention estimates for different outcomes. Trial follow-up differs, and overlapping outcomes must not be added.
- **D:** Historical and later mortality estimates alongside our recalculation. The reviews overlap and are compared, not pooled together.

Panels A–C redraw published results; D includes our independent calculation. An interval crossing 1 indicates uncertainty, not proof of zero benefit.


In [ ]:
plots.overview(summary)
display(Image(filename=str(PLOT_DIR / "statins_overview.png"), width=1200))


## How much does trial selection matter

The left plot shows the recalculated mortality effects. Larger squares carry more statistical weight. The right plot removes one contributing trial at a time; its thin teal line marks the full pooled estimate.

These omissions are **exploratory sensitivity checks**. They change the evidence set, not the treatment received by patients within a trial. Moving across a significance threshold does not itself demonstrate Simpson's paradox or prove an individual trial is biased.


In [ ]:
mortality_result = analyze_counts(mortality)
loo = leave_one_out(mortality)

plots.mortality_details(mortality, mortality_result, loo)
display(Image(filename=str(PLOT_DIR / "mortality_meta_analysis.png"), width=1200))


In [ ]:
sensitivity_results = sensitivity(mortality)
sensitivity_table = pd.DataFrame({
    "Analysis": sensitivity_results["analysis"],
    "Contributing estimates": sensitivity_results["trials"],
    "RR (95% CI)": [
        f"{row.rr:.3f} ({row.ci_low:.3f}–{row.ci_high:.3f})"
        for row in sensitivity_results.itertuples()
    ],
})
display(sensitivity_table)


## Harms in absolute terms

**Left:** Muscle symptoms occur in both treatment groups. The difference between them estimates the treatment-related excess. Rates are per 1,000 person-years; parentheses show 95% intervals for the excess.

**Right:** The published diabetes projections show greater absolute excess at higher baseline glycaemia. They assume a constant relative effect within each intensity category. The intensity categories involve different trials and ascertainment, so this is not a direct randomized dose comparison.

These analyses include both prevention settings. Their annual harm rates cannot simply be subtracted from the overview's five-year benefit projections.


In [ ]:
plots.harms_context()
display(Image(filename=str(PLOT_DIR / "harms_context.png"), width=1100))


## What we can conclude from this notebook

The recalculation reproduces the 2022 review's rounded pooled results. It supports a small mortality reduction in that evidence set, with uncertainty that changes under some trial omissions. The overview separately shows the evidence for preventing cardiovascular events before established disease.

CTT's participant data require [requests to trial custodians](https://www.cttcollaboration.org/about2), so its risk-adjusted models have not been independently refitted here. No regression through coarse risk categories was added: their exact risks and covariance are unavailable. This is a reconstruction of the cited evidence, **not a systematic review updated through 2026**.

The [analysis guide](README.md) gives fuller methods. PDF and SVG exports (`notebook_output/`, generated locally) are regenerated alongside the notebook figures.
